# 03 — Results & Model Comparison

Load the per-model metrics produced by `src/evaluate.py` and build the comparison **table**,
**confusion-matrix grid**, and **ROC curves**. Run `evaluate.py` for each trained model first:

```bash
python src/evaluate.py --config config.yaml --model custom_cnn
python src/evaluate.py --config config.yaml --model resnet50
python src/evaluate.py --config config.yaml --model efficientnet_b0
python src/evaluate.py --config config.yaml --model densenet121
```

> ⚠️ Research/education only — not a medical device.


In [ ]:
import sys, os, json, glob; sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import load_config
cfg=load_config('../config.yaml'); CLASS_NAMES=cfg['classes']['names']
REPORTS='../reports'; FIG='../reports/figures'


## 1. Comparison table
We rank by **Quadratic Weighted Kappa (QWK)** — the standard DR metric — and also show accuracy,
macro-F1, macro ROC-AUC, and **referable-DR recall** (how many grade≥2 patients we catch).


In [ ]:
rows=[]
for path in sorted(glob.glob(f'{REPORTS}/metrics_*.json')):
    m=json.load(open(path))
    rows.append({
        'model': m['model'],
        'QWK': round(m['qwk'],3),
        'accuracy': round(m['accuracy'],3),
        'macro_F1': round(m['macro_f1'],3),
        'macro_ROC_AUC': round(m['macro_roc_auc'],3) if m.get('macro_roc_auc') else None,
        'referable_recall': round(m['referable']['recall'],3),
    })
if rows:
    table=pd.DataFrame(rows).sort_values('QWK', ascending=False).reset_index(drop=True)
    display(table)
    table.to_csv(f'{REPORTS}/comparison_table.csv', index=False)
    print('Best model by QWK:', table.iloc[0]['model'])
else:
    print('No metrics_*.json found yet. Run src/evaluate.py for each model first.')


## 2. Confusion-matrix grid
Rows = true grade, columns = predicted. A good model has a strong diagonal. Watch the rare severe
grades — that's where imbalance bites.


In [ ]:
paths=sorted(glob.glob(f'{REPORTS}/metrics_*.json'))
if paths:
    n=len(paths); fig,axes=plt.subplots(1,n,figsize=(5*n,4.2)); axes=np.atleast_1d(axes)
    import seaborn as sns
    for ax,p in zip(axes,paths):
        m=json.load(open(p)); cm=np.array(m['confusion_matrix'])
        sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax,
                    xticklabels=range(len(CLASS_NAMES)),yticklabels=range(len(CLASS_NAMES)))
        ax.set_title(m['model']); ax.set_xlabel('pred'); ax.set_ylabel('true')
    fig.suptitle('Confusion matrices',fontweight='bold'); fig.tight_layout()
    fig.savefig(f'{FIG}/confusion_grid.png',dpi=140,bbox_inches='tight'); plt.show()
else: print('Run evaluate.py first.')


## 3. ROC curves
`evaluate.py` already saved per-model ROC figures to `reports/figures/roc_curves_<model>.png`.
Display them inline here for the report.


In [ ]:
from PIL import Image
roc=sorted(glob.glob(f'{FIG}/roc_curves_*.png'))
for p in roc:
    plt.figure(figsize=(5,5)); plt.imshow(Image.open(p)); plt.axis('off'); plt.title(os.path.basename(p)); plt.show()


## 4. Results narrative (template — fill in with your numbers)

_Replace the bracketed parts after running on real APTOS data._

**Which model wins?** Ranked by QWK, **[best model]** led with QWK **[0.xx]**, ahead of the
from-scratch custom CNN (QWK **[0.xx]**). This is the expected pattern: transfer-learning models
start from ImageNet features and need far less data to reach strong performance, while the custom
CNN (~0.39M params, no pretraining) is a useful but weaker baseline.

**Why accuracy alone is misleading.** Because ~49% of images are 'No DR', a trivial
always-predict-0 model scores ~49% accuracy yet has **zero** recall on sick patients. That's why
we lead with QWK and watch **referable-DR recall** (grade≥2): catching patients who need referral
matters more than raw accuracy.

**Trade-offs.** EfficientNet-B0 typically gives the best accuracy-per-parameter (cheap to train);
ResNet50 is a robust, well-understood baseline; DenseNet121 is parameter-efficient via feature
reuse. The confusion matrices usually show adjacent-grade confusion (e.g. 1↔2) — expected for an
ordinal task and exactly what QWK forgives more gently than far-off errors.

_Next: Grad-CAM explainability._
